In [1]:
import os
import shutil
import yaml

images_dir = r"D:\gradproject\dataset\train\images"
labels_dir = r"D:\gradproject\dataset\train\labels"
output_dir = r"D:\gradproject\dataset_converted"

with open(r"D:\gradproject\dataset\data.yaml", "r") as f:
    data = yaml.safe_load(f)

class_names = data["names"]

os.makedirs(output_dir, exist_ok=True)

for label_file in os.listdir(labels_dir):
    label_path = os.path.join(labels_dir, label_file)

    with open(label_path, "r") as f:
        line = f.readline().strip()

    if not line:
        continue

    class_id = int(line.split()[0])
    class_name = class_names[class_id]

    class_folder = os.path.join(output_dir, class_name)
    os.makedirs(class_folder, exist_ok=True)

    image_name = label_file.replace(".txt", ".jpg")
    src_image = os.path.join(images_dir, image_name)
    dst_image = os.path.join(class_folder, image_name)

    if os.path.exists(src_image):
        shutil.copy(src_image, dst_image)

print("Done!")

Done!


SyntaxError: invalid syntax (4214150504.py, line 1)

In [1]:
import os
import cv2
import numpy as np
from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

# ======================
# 📌 Preprocessing
# ======================
def process_image(path):
    img = cv2.imread(path)

    if img is None:
        raise ValueError("Invalid image")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    return img

# ======================
# 📌 Feature Extraction (🔥 محسنة)
# ======================
def extract_features(image):

    # 1. RGB stats
    mean = image.mean(axis=(0,1))
    std = image.std(axis=(0,1))

    # 2. HSV stats
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    hsv_mean = hsv.mean(axis=(0,1))
    hsv_std = hsv.std(axis=(0,1))

    # 3. LBP (texture)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method="uniform")

    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    # 4. Edges
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size

    # 5. Color Histogram (🔥 مهم جدًا للتمييز)
    hsv_hist = cv2.calcHist([hsv], [0,1], None, [8,8], [0,180,0,256])
    hsv_hist = cv2.normalize(hsv_hist, hsv_hist).flatten()

    # 6. White pixels (تمييز mildew)
    white_pixels = np.sum(
        (image[:,:,0] > 200) &
        (image[:,:,1] > 200) &
        (image[:,:,2] > 200)
    ) / image.size

    return np.concatenate([
        mean, std,
        hsv_mean, hsv_std,
        hist,
        hsv_hist,
        [edge_density],
        [white_pixels]
    ])

# ======================
# 📌 Load Dataset
# ======================
dataset_path = r"D:\gradproject\dataset_converted"

X = []
y = []

class_names = os.listdir(dataset_path)
label_map = {name: idx for idx, name in enumerate(class_names)}

for class_name in class_names:
    folder = os.path.join(dataset_path, class_name)

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        try:
            img = process_image(path)
            features = extract_features(img)

            X.append(features)
            y.append(label_map[class_name])
        except:
            continue

X = np.array(X)
y = np.array(y)

# ======================
# 📌 Scaling (🔥 مهم جدًا)
# ======================
scaler = StandardScaler()
X = scaler.fit_transform(X)

# ======================
# 📌 Train Model (🔥 أقوى)
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# ======================
# 📌 Evaluation
# ======================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ======================
# 📌 Save Model
# ======================
joblib.dump(model, "model.pkl")
joblib.dump(label_map, "label_map.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model saved ✅")

Accuracy: 0.9848901098901099
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       144
           1       1.00      0.95      0.98        22
           2       1.00      1.00      1.00        57
           3       0.94      0.99      0.97       148
           4       1.00      1.00      1.00        38
           5       1.00      0.98      0.99       154
           6       1.00      0.84      0.91        31
           7       1.00      0.99      1.00       134

    accuracy                           0.98       728
   macro avg       0.99      0.97      0.98       728
weighted avg       0.99      0.98      0.98       728

Model saved ✅


In [13]:
import os
import cv2
import numpy as np
from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# ======================
# 📌 Preprocessing
# ======================
def process_image(path):
    img = cv2.imread(path)

    if img is None:
        raise ValueError("Invalid or unsupported image format")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    return img

# ======================
# 📌 Feature Extraction
# ======================
def extract_features(image):
    import cv2
    import numpy as np
    from skimage.feature import local_binary_pattern

    # 1. Color
    mean = image.mean(axis=(0,1))
    std = image.std(axis=(0,1))

    # 2. HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    hsv_mean = hsv.mean(axis=(0,1))
    hsv_std = hsv.std(axis=(0,1))

    # 3. LBP
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method="uniform")

    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    # 4. Edges
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size

    return np.concatenate([
        mean, std,
        hsv_mean, hsv_std,
        hist,
        [edge_density]
    ])

# ======================
# 📌 Load Dataset
# ======================
dataset_path = r"D:\gradproject\dataset_converted"

X = []
y = []

class_names = os.listdir(dataset_path)
label_map = {name: idx for idx, name in enumerate(class_names)}

for class_name in class_names:
    folder = os.path.join(dataset_path, class_name)

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        try:
            img = process_image(path)
            features = extract_features(img)

            X.append(features)
            y.append(label_map[class_name])
        except:
            continue

X = np.array(X)
y = np.array(y)

# ======================
# 📌 Train Model
# ======================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# ======================
# 📌 Evaluation
# ======================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.945054945054945
              precision    recall  f1-score   support

           0       0.97      0.93      0.95       117
           1       0.96      0.73      0.83        33
           2       1.00      0.98      0.99        47
           3       0.86      0.98      0.92       137
           4       0.96      0.93      0.95        28
           5       0.95      0.98      0.96       168
           6       0.93      0.76      0.84        34
           7       0.98      0.96      0.97       164

    accuracy                           0.95       728
   macro avg       0.95      0.91      0.93       728
weighted avg       0.95      0.95      0.94       728



In [2]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(label_map, "label_map.pkl")

print("Model saved ✅")

Model saved ✅


In [3]:
def predict_image(path):
    img = process_image(path)
    features = extract_features(img)

    pred = model.predict([features])[0]

    reverse_map = {v:k for k,v in label_map.items()}
    return reverse_map[pred]

In [4]:
treatments = {
    "Angular Leafspot": "Use copper fungicide",
    "Anthracnose Fruit Rot": "Remove infected fruits",
    "Blossom Blight": "Apply fungicide early",
    "Gray Mold": "Improve ventilation",
    "Leaf Spot": "Use resistant plants",
    "Powdery Mildew Fruit": "Apply sulfur spray",
    "Powdery Mildew Leaf": "Avoid humidity",
    "healthy": "Plant is healthy, no treatment needed"
}

In [5]:
result = predict_image(r"D:\gradproject\testhealthy.jpg")

print("Disease:", result)
print("Treatment:", treatments[result])

Disease: Gray Mold
Treatment: Improve ventilation


In [6]:
!!pip install opencv-python scikit-image scikit-learn numpy

['Requirement already satisfied: opencv-python in c:\\users\\dell\\appdata\\local\\programs\\python\\python311\\lib\\site-packages (4.13.0.92)',
 'Collecting scikit-image',
 '  Downloading scikit_image-0.26.0-cp311-cp311-win_amd64.whl.metadata (15 kB)',
 'Requirement already satisfied: scikit-learn in c:\\users\\dell\\appdata\\local\\programs\\python\\python311\\lib\\site-packages (1.8.0)',
 'Requirement already satisfied: numpy in c:\\users\\dell\\appdata\\local\\programs\\python\\python311\\lib\\site-packages (2.2.1)',
 'Requirement already satisfied: scipy>=1.11.4 in c:\\users\\dell\\appdata\\local\\programs\\python\\python311\\lib\\site-packages (from scikit-image) (1.17.0)',
 'Collecting networkx>=3.0 (from scikit-image)',
 '  Downloading networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)',
 'Requirement already satisfied: pillow>=10.1 in c:\\users\\dell\\appdata\\local\\programs\\python\\python311\\lib\\site-packages (from scikit-image) (11.1.0)',
 'Collecting imageio!=2.35.0,>=2.

In [4]:
!!pip install opencv-python

['Collecting opencv-python',
 '  Downloading opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)',
 'Requirement already satisfied: numpy>=2 in c:\\users\\dell\\appdata\\local\\programs\\python\\python311\\lib\\site-packages (from opencv-python) (2.2.1)',
 'Downloading opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)',
 '   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--',
 '    --------------------------------------- 0.5/40.2 MB 3.4 MB/s eta 0:00:12',
 '   - -------------------------------------- 1.3/40.2 MB 3.7 MB/s eta 0:00:11',
 '   -- ------------------------------------- 2.1/40.2 MB 3.9 MB/s eta 0:00:10',
 '   -- ------------------------------------- 2.9/40.2 MB 3.9 MB/s eta 0:00:10',
 '   --- ------------------------------------ 3.9/40.2 MB 3.9 MB/s eta 0:00:10',
 '   ---- ----------------------------------- 4.7/40.2 MB 3.9 MB/s eta 0:00:10',
 '   ----- ---------------------------------- 5.5/40.2 MB 3.9 MB/s eta 0:00:09',
 '   ------ ----